In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from einops import rearrange

from genpp.models.loss import MultiPatchwiseRBFScore, PatchwiseEnergyScore, PatchwiseRBFScore

In [6]:
y = torch.randn(10, 2, 30, 30)  # [batch, variables, height, width]
x = torch.randn(10, 2, 10, 30, 30) * 1.0 + y.unsqueeze(
    2
)  # [batch, variables, n_samples, height, width]

patch_size = 5
rbf_score = PatchwiseRBFScore(
    patch_size=patch_size,
    lengthscales=torch.Tensor([0.5, 1.0, 2.0]) * patch_size**2,
    height=30,
    width=30,
)

x_per_var = rearrange(x, "b c n h w -> b n (c h w)")
y_per_var = rearrange(y, "b c h w -> b (c h w)")

s2 = rbf_score.forward(x_per_var, y_per_var, mode="complete")
print(s2.shape)
print(s2.mean())

torch.Size([10])
tensor(-0.1542)


In [11]:
for patch_size in range(3, 15, 2):
    es = PatchwiseEnergyScore(patch_size=patch_size, height=30, width=30, normalize=True)
    rbf_score = PatchwiseRBFScore(
        patch_size=patch_size, lengthscales=patch_size**2.0, height=30, width=30
    )

    x = torch.randn(10, 2, 10, 30, 30)  # [batch, variables, n_samples, height, width]
    y = torch.randn(10, 2, 30, 30)  # [batch, variables, height, width]

    x_per_var = rearrange(x, "b c n h w -> b n (c h w)")
    y_per_var = rearrange(y, "b c h w -> b (c h w)")

    s1 = es.forward(x_per_var, y_per_var, mode="complete")
    s2 = rbf_score.forward(x_per_var, y_per_var, mode="complete")
    print(f"Patch size: {patch_size}")
    print(s1.mean())
    print(s2.mean())

Patch size: 3
tensor(1.0760)
tensor(-0.0196)
Patch size: 5
tensor(1.0917)
tensor(-0.0128)
Patch size: 7
tensor(1.0998)
tensor(-0.0107)
Patch size: 9
tensor(1.0991)
tensor(-0.0101)
Patch size: 11
tensor(1.0993)
tensor(-0.0100)
Patch size: 13
tensor(1.1124)
tensor(-0.0087)


In [16]:
mrfb = MultiPatchwiseRBFScore(
    patch_sizes=[3, 9, 13],
    lengthscales=[[0.5, 1.0, 2.0], torch.Tensor([0.1, 0.5, 1.0]), 13**2.0],
    height=30,
    width=30,
)

y = torch.randn(10, 2, 30, 30)  # [batch, variables, height, width]
x = torch.randn(10, 2, 10, 30, 30) * 0.01 + y.unsqueeze(
    2
)  # [batch, variables, n_samples, height, width]

x_per_var = rearrange(x, "b c n h w -> b n (c h w)")
y_per_var = rearrange(y, "b c h w -> b (c h w)")

s = mrfb.forward(x_per_var, y_per_var, mode="complete")
print("Multi patch sizes:")
print(s.mean())

Multi patch sizes:
tensor(-0.4987)
